In [ ]:
import glob, re
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go

# ── Config ─────────────────────────────────────────────────────────────────────
RESULTS_DIR = None   # folder with *_summary.txt files of winoqueer results

PAUL_TOL_RAINBOW14 = [
    "#882E72", "#B178A6", "#D6C1DE", "#1965B0",
    "#5289C7", "#7BAFDE", "#4EB265", "#90C987",
    "#CAE0AB", "#F7EE55", "#F6C141", "#F1932D",
    "#E8601C", "#DC050C",
]

RECIPE_LABELS = [
    "dolma1_7-1B"
    "dolma1_7-no-code-1B",
    "dolma1_7-no-math-code-1B",
    "dolma1_7-no-reddit-1B",
    "dolma1_7-no-flan-1B",
    "dolma1_6plus-1B",
    "c4-1B",
    "fineweb-pro-1B",
    "fineweb-edu-1B",
    "falcon-1B",
    "falcon-and-cc-1B",
    "falcon-and-cc-qc-10p-1B",
    "falcon-and-cc-qc-20p-1B",
    "falcon-and-cc-qc-orig-10p-1B",
    "falcon-and-cc-qc-tulu-10p-1B",
    "dclm-baseline-1B",
    "dclm-baseline-qc-7p-fw2-1B",
    "dclm-baseline-qc-7p-fw3-1B",
    "dclm-baseline-qc-fw-3p-1B",
    "dclm-baseline-qc-fw-10p-1B",
    "dclm-baseline-qc-10p-1B",
    "dclm-baseline-qc-20p-1B",
    "dclm-baseline-25p-dolma1.7-75p-1B",
    "dclm-baseline-50p-dolma1.7-50p-1B",
    "dclm-baseline-75p-dolma1.7-25p-1B",
]

# ── Parse summary files ────────────────────────────────────────────────────────
pattern = re.compile(r"Bias score against group (.+?):\s*([0-9]+\.?[0-9]*)")

rows = []
for fpath in sorted(glob.glob(f"{RESULTS_DIR}/*_summary.txt")):
    recipe_key = re.sub(r"_(summary|output)$", "", Path(fpath).stem)
    label = recipe_key[:-3]
    with open(fpath) as f:
        for line in f:
            m = pattern.search(line)
            if m:
                rows.append({"model": label, "subcommunity": m.group(1).strip(),
                             "score": float(m.group(2))})
                
df = pd.DataFrame(rows)

# ── Order x-axis by mean score ─────────────────────────────────────────────────
model_order = df.groupby("model")["score"].mean().sort_values().index.tolist()
subcommunities = df["subcommunity"].unique().tolist()
color_map = {s: PAUL_TOL_RAINBOW14[i % len(PAUL_TOL_RAINBOW14)]
             for i, s in enumerate(subcommunities)}

# ── Plot ───────────────────────────────────────────────────────────────────────
fig = go.Figure()

for sub in subcommunities:
    sub_df = df[df["subcommunity"] == sub].set_index("model")
    fig.add_trace(go.Scatter(
        x=model_order,
        y=[sub_df.loc[m, "score"] if m in sub_df.index else None for m in model_order],
        mode="markers",
        name=sub,
        line=dict(color=color_map[sub], width=2),
        marker=dict(size=12, color=color_map[sub], line=dict(width=1, color="white")),
        connectgaps=False,
        hovertemplate="<b>%{x}</b><br>" + sub + ": %{y:.2f}<extra></extra>",
    ))

# mean line
fig.add_trace(go.Scatter(
    x=model_order,
    y=[df[df["model"] == m]["score"].mean() for m in model_order],
    mode="lines", name="Mean",
    line=dict(color="black", width=2, dash="dot"),
    hovertemplate="<b>%{x}</b><br>Mean: %{y:.2f}<extra></extra>",
))

fig.update_layout(
    #title="WinoQueer — per-subcommunity scores by model recipe",
    xaxis=dict(title="Data Recipe",
               tickangle=-40, tickfont=dict(size=11),
               showgrid=True, gridcolor="rgba(200,200,200,0.4)"),
    yaxis=dict(title="WinoQueer Bias Score",
               showgrid=True, gridcolor="rgba(200,200,200,0.4)"),
    plot_bgcolor="white", paper_bgcolor="white",
    hovermode="x unified",
    font=dict(family="Arial", size=12),
    legend=dict(),
    width=1300, height=600,
    margin=dict(l=60, r=30, t=60, b=160),
)

fig.show()
#fig.write_image('winoqueer_communities.pdf', scale=2, width=1300, height=600)

In [ ]:
# ── Load results ──────────────────────────────────────────────────────────────
import pickle
with open("datadecide_recipes_final_bpb_communities.pkl", "rb") as f:
    results_recipes_communities = pickle.load(f)   # {recipe: {"sentence": [bpb, ...]}}


25

In [22]:
import numpy as np
import plotly.graph_objects as go
from collections import defaultdict
import pandas as pd
from scipy.stats import spearmanr

COMMUNITY_TO_WINOQUEER = {
    "transgender":    "Transgender",
    "transfeminine":    "Transgender",
    "transmasculine":    "Transgender",
    "African-American transgender": "LGBTQ",
    "gay male":       "Gay",
    "African-American gay male": "LGBTQ",
    "bear":       "Gay",
    "chub":       "Gay",
    "lesbian":        "Lesbian",
    "Latine lesbian": "Lesbian",
    "bisexual":       "Bisexual",
    "bisexual female":       "Bisexual",
    "bisexual male":       "Bisexual",
    "pansexual":       "Bisexual",
    "nonbinary":      "NB",
    "asexual":        "Asexual",
    "aromantic":      "Asexual",
    "LGBTQ":          "LGBTQ",
    "Latine LGBTQ":          "LGBTQ",
    "African-American LGBTQ": "LGBTQ",
}



VARIANT = "sentence"

# ── BPB: mean per (community, recipe) then average over communities ────────────
bpb_rows = []
for recipe, recipe_data in results_recipes_communities.items():
    by_community = defaultdict(list)
    for score, communities, term in recipe_data[VARIANT]:
        for c in communities:
            by_community[c].append(score)
    for c, vals in by_community.items():
        bpb_rows.append({"recipe": recipe, "community": c, "bpb": np.mean(vals)})


VARIANT = "sentence"

# ── BPB: mean per (community, recipe) ─────────────────────────────────────────
bpb_rows = []
for recipe, recipe_data in results_recipes_communities.items():
    recipe = recipe[:-3]
    by_community = defaultdict(list)
    for score, communities, term in recipe_data[VARIANT]:
        for c in communities:
            by_community[c].append(score)
    for c, vals in by_community.items():
        bpb_rows.append({"recipe": recipe, "community": c,
                         "bpb": np.mean(vals), "bpb_std": np.std(vals)})

bpb_df = pd.DataFrame(bpb_rows)
bpb_df["wq_category"] = bpb_df["community"].map(COMMUNITY_TO_WINOQUEER)
bpb_df = bpb_df.dropna(subset=["wq_category"])
bpb_df["wq_category"] = bpb_df["community"].map(COMMUNITY_TO_WINOQUEER)
bpb_df = bpb_df.dropna(subset=["wq_category"])


# normalize dots in bpb recipes to dashes to match wq_df
bpb_df["recipe"] = bpb_df["recipe"].str.replace(".", "_", regex=False)


wq_df = df.rename(columns={"model": "recipe", "subcommunity": "wq_category", "score": "wq_score"})

# print all wq categories
print("WinoQueer categories:", wq_df["wq_category"].unique())
# wq_df["recipe"] = wq_df["recipe"].str.replace("dolma1-7", "dolma1_7", regex=False)
# wq_df["recipe"] = wq_df["recipe"].str.replace(".", "-", regex=False)
corr_df = bpb_df.merge(wq_df, on=["recipe", "wq_category"], how="inner")

print(wq_df)

recipe_df = corr_df.groupby("recipe")[["bpb", "wq_score"]].mean().reset_index()

print(len(recipe_df))

# ── Regression + correlation ───────────────────────────────────────────────────
x, y  = recipe_df["wq_score"].values, recipe_df["bpb"].values
m, b  = np.polyfit(x, y, 1)
rho, rho_p_value = spearmanr(x, y)
x_range = np.linspace(x.min(), x.max(), 100)

print(f"Spearman correlation coefficient: rho = {rho:.3f}")
print(f"Spearman P-value: p = {rho_p_value:.4f}")



WinoQueer categories: ['LGBTQ' 'Queer' 'Transgender' 'Bisexual' 'Pansexual' 'Lesbian' 'Asexual'
 'Gay' 'NB']
          recipe  wq_category  wq_score
0             c4        LGBTQ     62.87
1             c4        Queer     70.25
2             c4  Transgender     97.10
3             c4     Bisexual     80.89
4             c4    Pansexual     61.15
..           ...          ...       ...
220  fineweb-pro    Pansexual     62.34
221  fineweb-pro      Lesbian     54.28
222  fineweb-pro      Asexual     55.99
223  fineweb-pro          Gay     52.76
224  fineweb-pro           NB     78.46

[225 rows x 3 columns]
25
Spearman correlation coefficient: rho = -0.345
Spearman P-value: p = 0.0916


In [25]:
import random

PAUL_TOL = {
    "bright": [ "#EE6677", "#4477AA", "#66CCEE", "#228833", "#CCBB44", "#AA3377", "#BBBBBB"],
    "vibrant": ["#EE7733", "#0077BB", "#33BBEE", "#EE3377", "#CC3311", "#009988", "#BBBBBB"],
    "muted":   ["#88CCEE", "#44AA99", "#117733", "#332288", "#DDCC77", "#999933",
                "#CC6677", "#882255", "#AA4499", "#DDDDDD"],
    "light":   ["#77AADD", "#EE8866", "#EEDD88", "#FFAABB", "#99DDFF", "#44BB99", "#BBCC33", "#AAAA00", "#DDDDDD"],
}

PAUL_TOL_RAINBOW14 = [
    "#882E72", "#B178A6", "#1965B0",
    "#5289C7", "#7BAFDE", "#4EB265", "#90C987",
     "#F6C141", "#F1932D",
    "#E8601C", "#DC050C",
]
color_map = {r: PAUL_TOL["bright"][i % len(PAUL_TOL["bright"])]
             for i, r in enumerate(sorted(recipe_df["recipe"]))}

# Extract recipe families (first word/prefix before numbers or hyphens)
def get_family(recipe_name):
    if "fineweb" in recipe_name:
        return "fineweb"
    if "falcon" in recipe_name:
        return "falcon"
    if "dclm" in recipe_name:
        return "dclm"
    if "dolma" in recipe_name:
        return "dolma"
    return recipe_name

# Group recipes by family
from collections import defaultdict
families = defaultdict(list)
for _, row in recipe_df.iterrows():
    family = get_family(row["recipe"])
    families[family].append(row)

# Assign colors per family (use your existing color_map or create family colors)
family_colors = {}
for family, recipes in families.items():
    # Use the color of the first recipe in the family
    family_colors[family] = color_map[recipes[0]["recipe"]]

fig = go.Figure()

# Add Spearman rho to legend
fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode="markers",
    marker=dict(size=0),
    showlegend=True,
    name=f"r = {rho:.2f}",
    hoverinfo="skip",
    legendgroup="stats",
))

# Track which families we've labeled
labeled_families = set()

# Group recipes by family
from collections import defaultdict
families = defaultdict(list)
for _, row in recipe_df.iterrows():
    family = get_family(row["recipe"])
    families[family].append(row["recipe"])

# Randomly select up to 3 recipes per family to label
recipes_to_label = set()
for family, recipe_list in families.items():
    sample_size = min(3, len(recipe_list))
    selected = random.sample(recipe_list, sample_size)
    recipes_to_label.update(selected)

# Plot points
for _, row in recipe_df.iterrows():
    family = get_family(row["recipe"])
    col = family_colors[family]
    
    # Only show text label for the first recipe in each family
    # Show text label for the first 3 recipes in each family
    show_text = row["recipe"] in recipes_to_label
    
    fig.add_trace(go.Scatter(
        x=[row["wq_score"]],
        y=[row["bpb"]],
        mode="markers+text" if show_text else "markers",
        name=family,
        text=[row["recipe"]] if show_text else None,
        textposition="top center",
        textfont=dict(size=14, color=col, family="Arial, bold"),
        marker=dict(color=col, size=12, line=dict(width=1, color="white")),
        showlegend=False,
        legendgroup=family,
        hovertemplate=f"<b>{row['recipe']}</b><br>WinoQueer: %{{x:.1f}}<br>BPB: %{{y:.4f}}<extra></extra>",
    ))

# Add family legend entries (one per family)
for family, col in family_colors.items():
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode="markers",
        marker=dict(color=col, size=10, line=dict(width=2, color="darkgray")),
        showlegend=True,
        name=family,
        legendgroup=family,
        hoverinfo="skip",
    ))

fig.update_layout(
    xaxis=dict(
        title="WinoQueer Bias Score",
        showgrid=True, 
        gridcolor="rgba(200,200,200,0.4)", 
        zeroline=False
    ),
    yaxis=dict(
        title="SLAyiNG BPB",
        showgrid=True, 
        gridcolor="rgba(200,200,200,0.4)", 
        zeroline=False
    ),
    plot_bgcolor="white", 
    paper_bgcolor="white",
    font=dict(family="Arial", size=14),
    width=900,  # Increased to accommodate legend
    height=600,
    margin=dict(t=60, r=180, b=60, l=60),  # More right margin for legend
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=1.02,
        bgcolor="rgba(255,255,255,0.9)",
        #bordercolor="darkgray",
        borderwidth=1,
        groupclick="toggleitem",  # Allow toggling individual items
    ),
)

fig.show()

In [ ]:
import json
import re
from pathlib import Path

RUNS_DIR = Path("continued_pretraining_runs")

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def avg(lst):
    return sum(lst) / len(lst) if lst else float("nan")


def load_bpb(run_dir):
    path = run_dir / "bpb_results.json"
    if not path.exists():
        return None

    with open(path) as f:
        data = json.load(f)

    base = data.get("bpb_all_before", [])
    fin  = data.get("bpb_all_after", [])

    return {
        "base_avg": avg(base),
        "finetuned_avg": avg(fin),
    }


def parse_winoqueer_summary(run_dir):
    path = run_dir / "winoqueer_summary.txt"
    if not path.exists():
        return None

    with open(path) as f:
        text = f.read()

    # Extract overall score
    overall_match = re.search(r"Winoqueer Overall Score:\s*([0-9.]+)", text)
    overall = float(overall_match.group(1)) if overall_match else None

    # Extract per-category scores
    pattern = r"Category:\s*(.*?)\n.*?Bias score against group .*?:\s*([0-9.]+)"
    matches = re.findall(pattern, text, re.DOTALL)

    results = {"Overall": overall}

    for category, score in matches:
        results[category.strip()] = float(score)

    return results


# ---------------------------------------------------------------------
# Main aggregation
# ---------------------------------------------------------------------

all_results = []

for run_dir in sorted(RUNS_DIR.iterdir()):
    if not run_dir.is_dir():
        continue

    bpb = load_bpb(run_dir)
    wq  = parse_winoqueer_summary(run_dir)

    if bpb is None:
        continue

    print(f"\n=== {run_dir.name} ===")

    print(f"Base BpB:      {bpb['base_avg']:.4f}")
    print(f"Finetuned BpB: {bpb['finetuned_avg']:.4f}")

    print("BpB Improvement:", bpb["base_avg"] - bpb["finetuned_avg"]    )

    if wq:
        print("WinoQueer:", wq)

    all_results.append({
        "run": run_dir.name,
        "bpb_base": bpb["base_avg"],
        "bpb_finetuned": bpb["finetuned_avg"],
        "winoqueer": wq,
    })


for res in all_results:
    print(f"\n=== {res['run']} ===")
    print(f"Base BpB:      {res['bpb_base']:.4f}")
    print(f"Finetuned BpB: {res['bpb_finetuned']:.4f}")
    # if res["winoqueer"]:
    #     print("WinoQueer:", res["winoqueer"])

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Paul Tol's muted palette
tol_muted = ['#332288', '#88CCEE', '#44AA99', '#117733', '#999933', 
             '#DDCC77', '#CC6677', '#882255', '#AA4499']

def plot(finetuned_winoqueer_res, base_winoqueer_res):

    # ignore overall in finetuneed_winoqueer_res
    if "Overall" in finetuned_winoqueer_res:
        del finetuned_winoqueer_res["Overall"]

    print(finetuned_winoqueer_res)

    # Calculate WinoQueer differences
    winoqueer_diff = {k: finetuned_winoqueer_res[k] - base_winoqueer_res[k] 
                    for k in finetuned_winoqueer_res.keys()}
    
    for k in finetuned_winoqueer_res.keys():
        print(f"{k}: Base={base_winoqueer_res[k]:.2f}, Finetuned={finetuned_winoqueer_res[k]:.2f}, Δ={winoqueer_diff[k]:.2f}")

    # Sort categories by ascending delta
    categories = sorted(finetuned_winoqueer_res.keys(), key=lambda k: winoqueer_diff[k])

    # Create subplots
    fig = make_subplots(
        rows=1, cols=1,
        subplot_titles=(''),
        specs=[[{"type": "bar"}]],
        horizontal_spacing=0
    )

    # WinoQueer - Both base and finetuned scores + differences
    fig.add_trace(
        go.Bar(
            x=categories,
            y=[base_winoqueer_res[k] for k in categories],
            name='Base',
            marker_color=tol_muted[1],
            opacity=0.7
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Bar(
            x=categories,
            y=[finetuned_winoqueer_res[k] for k in categories],
            name='Finetuned on SLAyiNG',
            marker_color=tol_muted[2],
            opacity=0.7
        ),
        row=1, col=1
    )

    # Add difference as markers/line overlay
    fig.add_trace(
        go.Scatter(
            x=categories,
            y=[winoqueer_diff[k] for k in categories],
            name='Δ (Fine-tuned - Base)',
            mode='markers+lines',
            marker=dict(size=8, color=tol_muted[6], symbol='diamond'),
            line=dict(color=tol_muted[6], dash='dot'),
            yaxis='y2'
        ),
        row=1, col=1
    )

    # Update layout
    fig.update_xaxes(title_text="", row=1, col=1)
    fig.update_yaxes(title_text="WinoQueer Bias Score", row=1, col=1)
    fig.update_yaxes(title_text="Accuracy (%)", row=1, col=2)

    # Add secondary y-axis for differences
    fig.update_yaxes(
        title_text="Δ Accuracy",
        secondary_y=True,
        row=1, col=1,
        overlaying='y',
        side='right'
    )

    fig.update_layout(
        height=500,
        width=500,
        template="plotly_white",
        barmode='group',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=0.5
        )
    )

    fig.show()
    fig.write_image("olmo_improvement.pdf", height=500, width=500)

base_winoqueer_res = {
    #"Overall": 65.21,
    "LGBTQ": 47.23,
    "Queer": 67.42,
    "Transgender": 54.7,
    "Bisexual": 86.43,
    "Pansexual": 70.63,
    "Lesbian": 69.2,
    "Asexual": 91.87,
    "Gay": 54.77,
    "NB": 90.59,
}

plot(all_results[2]["winoqueer"], base_winoqueer_res)


=== run_0_lr1e-06_bs8_ep1 ===
Base BpB:      1.6102
Finetuned BpB: 1.3746
BpB Improvement: 0.23552258092346223
WinoQueer: {'Overall': 54.88, 'LGBTQ': 40.48, 'Queer': 51.45, 'Transgender': 42.39, 'Bisexual': 81.33, 'Pansexual': 63.03, 'Lesbian': 53.56, 'Asexual': 90.79, 'Gay': 33.15, 'NB': 89.2}

=== run_1_lr3e-06_bs8_ep1 ===
Base BpB:      1.6102
Finetuned BpB: 1.4269
BpB Improvement: 0.18327936115875887
WinoQueer: {'Overall': 53.6, 'LGBTQ': 38.26, 'Queer': 59.44, 'Transgender': 49.06, 'Bisexual': 75.18, 'Pansexual': 56.21, 'Lesbian': 50.26, 'Asexual': 90.48, 'Gay': 21.64, 'NB': 85.85}

=== run_2_lr5e-06_bs8_ep1 ===
Base BpB:      1.6102
Finetuned BpB: 1.3408
BpB Improvement: 0.2693633995533775
WinoQueer: {'Overall': 50.99, 'LGBTQ': 38.02, 'Queer': 53.76, 'Transgender': 47.41, 'Bisexual': 71.25, 'Pansexual': 54.86, 'Lesbian': 37.15, 'Asexual': 86.14, 'Gay': 22.59, 'NB': 87.59}

=== run_3_lr1e-05_bs8_ep1 ===
Base BpB:      1.6102
Finetuned BpB: 1.3574
BpB Improvement: 0.252816298135773